# 00 — Data Ingestion v1: Complete Feature Pipeline

End-to-end pipeline from raw OHLCV to a unified `BTCUSDT_1h_unified.parquet`.

## Pipeline Overview

| Phase | Output | Description |
|-------|--------|-------------|
| A | `data/external/fear_greed_index.parquet` | alternative.me daily Fear & Greed index |
| B | `data/external/coingecko_market_caps.parquet` | CoinGecko market cap & volume (~1yr) |
| C | `data/external/approx_market_caps.parquet` | Historical market caps via OHLCV × supply |
| D | `data/features/BTCUSDT_1h_features.parquet` | 198 V1 technical features |
| E | `data/features/BTCUSDT_1h_v3_features.parquet` | V3 external features **with lag fix** |
| F | `data/features/BTCUSDT_1h_v4_features.parquet` | V4 features (fractional diff, TFI, futures) |
| G | `data/features/BTCUSDT_1h_structural.parquet` | Structural features (swings, VWAP, MTF) |
| H | `data/features/BTCUSDT_1h_microstructure.parquet` | Causal microstructure (Roll, Amihud, SampEn) |
| I | `data/features/BTCUSDT_1h_unified.parquet` | **Merged output — load this in model notebooks** |
| J | — | Leakage validation (all features vs next-bar return) |
| K | `data/features/feature_registry_unified_v1.json` | Feature registry with metadata |

## Caching

Every phase checks if its output already exists. Set `FORCE_REBUILD = True` in Phase 0
to force a complete rebuild. Individual phases can be forced by temporarily deleting
the target parquet.

## Lag Fix (Phase E — Critical)

Daily-frequency external data (market cap dominance, Fear & Greed, etc.) is an **end-of-day**
snapshot. Without a 1-day forward shift, bar `D` would see the *current* day's market cap
change, leaking forward-looking information.

The `mkt_total_mcap_chg_24h` feature alone had `corr = +0.18` with next-bar returns before
this fix was applied. Phase E applies `.shift(1)` on every daily series BEFORE
`.reindex(btc.index, method='ffill')`.

## Prerequisites

Raw OHLCV parquets in `data/raw/` (from `00_data_ingestion_v0.ipynb`):
- `BTCUSDT_1h.parquet`, `ETHUSDT_1h.parquet`, ..., `LINKUSDT_1h.parquet`


---
## Phase 0 — Setup

Imports, paths, and helper constants.

In [37]:

from pathlib import Path
import numpy as np
import pandas as pd
import requests
import json
import time
import warnings
warnings.filterwarnings('ignore')

# ── Repo root ─────────────────────────────────────────────────────────────────
def _repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found — run from within the repo')

REPO       = _repo_root()
RAW_DIR    = REPO / 'data' / 'raw'
EXT_DIR    = REPO / 'data' / 'external'
FEAT_DIR   = REPO / 'data' / 'features'
STATIC_DIR = REPO / 'data' / 'static'
for d in [RAW_DIR, EXT_DIR, FEAT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Control ───────────────────────────────────────────────────────────────────
FORCE_REBUILD = False   # set True to regenerate all outputs

# ── Symbol lists ──────────────────────────────────────────────────────────────
SYMBOLS_1H = [
    'BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT',
    'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'LINKUSDT',
]
ALTS = [s for s in SYMBOLS_1H if s != 'BTCUSDT']

COINGECKO_IDS = {
    'BTCUSDT': 'bitcoin',     'ETHUSDT': 'ethereum',   'BNBUSDT': 'binancecoin',
    'XRPUSDT': 'ripple',      'SOLUSDT': 'solana',      'ADAUSDT': 'cardano',
    'DOGEUSDT': 'dogecoin',   'AVAXUSDT': 'avalanche-2','DOTUSDT': 'polkadot',
    'LINKUSDT': 'chainlink',  'USDTUSDT': 'tether',     'USDCUSDT': 'usd-coin',
}

# ── Cache helper ──────────────────────────────────────────────────────────────
def _skip(path: Path, label: str) -> bool:
    """Return True (and print a skip notice) when file exists and FORCE_REBUILD is False."""
    if not FORCE_REBUILD and path.exists():
        print(f'  SKIP {label}  ({path.stat().st_size/1e6:.1f} MB)')
        return True
    return False

print(f'Repo          : {REPO}')
print(f'FORCE_REBUILD : {FORCE_REBUILD}')
print(f'Symbols       : {SYMBOLS_1H}')


Repo          : /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system
FORCE_REBUILD : False
Symbols       : ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT', 'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'LINKUSDT']


---
## Phase A — Fear & Greed Index

Source: [alternative.me/fng/](https://alternative.me/fng/) — free, daily, from Feb 2018.

Fetches the full history (`limit=0`) in one request. Timestamps are Unix seconds → UTC dates.

Output: `data/external/fear_greed_index.parquet`


In [38]:

FNG_OUTPUT = EXT_DIR / 'fear_greed_index.parquet'

if _skip(FNG_OUTPUT, 'Fear & Greed Index'):
    pass
else:
    print('Phase A — Downloading Fear & Greed Index ...')
    try:
        resp = requests.get(
            'https://api.alternative.me/fng/',
            params={'limit': '0'},
            timeout=30,
        )
        resp.raise_for_status()
        fng_data = resp.json()['data']

        fng_df = pd.DataFrame(fng_data)
        fng_df['value'] = fng_df['value'].astype(int)
        fng_df['timestamp'] = pd.to_datetime(
            fng_df['timestamp'].astype(int), unit='s', utc=True
        )
        fng_df = fng_df.rename(
            columns={'timestamp': 'date', 'value_classification': 'classification'}
        )
        fng_df = fng_df[['date', 'value', 'classification']].sort_values('date').reset_index(drop=True)
        fng_df = fng_df.set_index('date')
        drop_cols = [c for c in ['time_until_update'] if c in fng_df.columns]
        fng_df = fng_df.drop(columns=drop_cols)

        fng_df.to_parquet(FNG_OUTPUT)
        print(f'  Downloaded {len(fng_df)} daily records')
        print(f'  Range: {fng_df.index.min().date()} → {fng_df.index.max().date()}')
        print(f'  Saved: {FNG_OUTPUT}')
    except Exception as e:
        print(f'  ERROR: {e}')


  SKIP Fear & Greed Index  (0.0 MB)


---
## Phase B — CoinGecko Market Caps

Source: CoinGecko free tier — last ~365 days of daily market cap + volume for each coin.

Sleeps 6.5 s between requests to respect the free-tier rate limit (~90 s total).

Output: `data/external/coingecko_market_caps.parquet`


In [39]:

CG_OUTPUT = EXT_DIR / 'coingecko_market_caps.parquet'

if _skip(CG_OUTPUT, 'CoinGecko market caps'):
    pass
else:
    print('Phase B — Downloading CoinGecko market caps (~90 s due to rate limits) ...')
    CG_BASE = 'https://api.coingecko.com/api/v3'
    CG_DAYS = 365
    all_records = []

    for symbol, cg_id in COINGECKO_IDS.items():
        print(f'  {cg_id} ({symbol}) ...', end=' ', flush=True)
        try:
            resp = requests.get(
                f'{CG_BASE}/coins/{cg_id}/market_chart',
                params={'vs_currency': 'usd', 'days': str(CG_DAYS)},
                timeout=30,
            )
            resp.raise_for_status()
            data = resp.json()
            prices = data.get('prices', [])
            mcaps  = data.get('market_caps', [])
            vols   = data.get('total_volumes', [])
            n = min(len(prices), len(mcaps), len(vols))
            for i in range(n):
                ts = pd.Timestamp(prices[i][0], unit='ms', tz='UTC')
                all_records.append({
                    'date': ts, 'symbol': symbol, 'cg_id': cg_id,
                    'price':        prices[i][1],
                    'market_cap':   mcaps[i][1] if mcaps[i][1] else np.nan,
                    'total_volume': vols[i][1]  if vols[i][1]  else np.nan,
                })
            print(f'{n} records')
            time.sleep(6.5)
        except Exception as e:
            print(f'ERROR: {e}')
            time.sleep(10)

    if all_records:
        cg_df = pd.DataFrame(all_records)
        cg_df.to_parquet(CG_OUTPUT, index=False)
        print(f'\n  Total: {len(cg_df)} records across {cg_df["symbol"].nunique()} coins')
        print(f'  Range: {cg_df["date"].min().date()} → {cg_df["date"].max().date()}')
        print(f'  Saved: {CG_OUTPUT}')
    else:
        print('  No data downloaded.')


  SKIP CoinGecko market caps  (0.1 MB)


---
## Phase C — Approximate Historical Market Caps

Extends CoinGecko history back to 2017 by multiplying daily close × circulating supply.
Circulating supply is a static snapshot from `data/static/crypto_market_caps.csv`.

Output: `data/external/approx_market_caps.parquet`


In [40]:

APPROX_OUTPUT = EXT_DIR / 'approx_market_caps.parquet'

if _skip(APPROX_OUTPUT, 'Approximate market caps'):
    pass
else:
    print('Phase C — Computing approximate market caps from OHLCV × supply ...')
    SUPPLY_PATH = STATIC_DIR / 'crypto_market_caps.csv'
    if not SUPPLY_PATH.exists():
        print(f'  ERROR: {SUPPLY_PATH} not found — skipping Phase C.')
    else:
        supply_df  = pd.read_csv(SUPPLY_PATH)
        supply_map = dict(zip(supply_df['symbol'], supply_df['circulating_supply']))
        print(f'  Loaded circulating supply for {len(supply_map)} coins')

        records = []
        for symbol in SYMBOLS_1H:
            p = RAW_DIR / f'{symbol}_1h.parquet'
            if not p.exists():
                print(f'  {symbol}: OHLCV missing, skipping')
                continue
            supply = supply_map.get(symbol, np.nan)
            if np.isnan(supply):
                print(f'  {symbol}: no supply data, skipping')
                continue
            df    = pd.read_parquet(p)
            if df.index.tz is None:
                df.index = df.index.tz_localize('UTC')
            daily = df['close'].resample('1D').last().dropna()
            for ts, price in daily.items():
                records.append({
                    'date':               ts,
                    'symbol':             symbol,
                    'close_price':        price,
                    'circulating_supply': supply,
                    'approx_market_cap':  price * supply,
                })
            print(f'  {symbol}: {len(daily)} daily records, supply={supply:,.0f}')

        if records:
            approx_df = pd.DataFrame(records)
            approx_df.to_parquet(APPROX_OUTPUT, index=False)
            print(f'\n  Total: {len(approx_df)} records')
            print(f'  Range: {approx_df["date"].min().date()} → {approx_df["date"].max().date()}')
            print(f'  Saved: {APPROX_OUTPUT}')


  SKIP Approximate market caps  (0.4 MB)


---
## Phase D — V1 Technical Features (198 features)

Computes all technical features directly from BTCUSDT 1h OHLCV. This is the largest phase.

Feature groups:
- Returns (log and pct-change) at 1h–168h horizons
- Volatility (rolling std, ATR, Garman-Klass, Parkinson)
- Moving averages (SMA/EMA 7–4320), Bollinger Bands, MACD, RSI, Stochastic
- Volume indicators (z-score, OBV, VWAP, MFI, CMF, AD)
- Candle patterns (doji, hammer, shooting star, engulfing, streaks)
- Ichimoku cloud (tenkan/kijun, cloud top/bottom, recency)
- Supertrend (3 ATR multipliers), Fibonacci retracements
- Pivot levels (previous-day H/L/C), round-number distances, breakouts
- Hurst exponent, skewness/kurtosis, autocorrelation, variance ratio
- Divergences (RSI, MACD, OBV vs price), MA cross signals, bull score
- Halving cycle (sin/cos/pos), time features, momentum coherence
- Trend score, rolling Sharpe, regime composite

Output: `data/features/BTCUSDT_1h_features.parquet`


In [41]:

V1_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_features.parquet'

if _skip(V1_OUTPUT, 'V1 technical features'):
    pass
else:
    print('Phase D — Building V1 technical features ...')

    btc = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet')
    btc.index = btc.index.tz_localize(None) if btc.index.tz else btc.index
    out = pd.DataFrame(index=btc.index)
    out['close'] = btc['close']

    # ── Returns ────────────────────────────────────────────────────────────────
    for h in [1, 2, 3, 6, 12, 24, 48, 72, 168]:
        out[f'ret_{h}h']     = btc['close'].pct_change(h)
        out[f'log_ret_{h}h'] = np.log(btc['close'] / btc['close'].shift(h))

    # ── Volatility (rolling std of log returns) ────────────────────────────────
    log_ret = np.log(btc['close'] / btc['close'].shift(1))
    for h in [6, 12, 24, 48, 72, 168]:
        out[f'vol_{h}h'] = log_ret.rolling(h).std()

    # ── ATR ───────────────────────────────────────────────────────────────────
    tr = pd.concat([
        btc['high'] - btc['low'],
        (btc['high'] - btc['close'].shift(1)).abs(),
        (btc['low']  - btc['close'].shift(1)).abs(),
    ], axis=1).max(axis=1)
    out['atr_14']      = tr.ewm(span=14, adjust=False).mean()
    out['atr_14_pct']  = out['atr_14'] / btc['close']
    out['atr_24']      = tr.ewm(span=24, adjust=False).mean()
    out['atr_24_pct']  = out['atr_24'] / btc['close']

    # ── Moving averages: close vs SMA/EMA at various windows ──────────────────
    for w in [7, 14, 20, 50, 100, 200, 336, 504, 720, 2160, 4320]:
        sma = btc['close'].rolling(w).mean()
        ema = btc['close'].ewm(span=w, adjust=False).mean()
        out[f'close_vs_sma_{w}'] = (btc['close'] - sma) / sma.replace(0, np.nan)
        out[f'close_vs_ema_{w}'] = (btc['close'] - ema) / ema.replace(0, np.nan)
    out['sma_200'] = btc['close'].rolling(200).mean()

    # ── Bollinger Bands ────────────────────────────────────────────────────────
    for w in [20, 50]:
        sma = btc['close'].rolling(w).mean()
        std = btc['close'].rolling(w).std()
        out[f'bb_width_{w}']    = (2 * std) / sma.replace(0, np.nan)
        out[f'bb_position_{w}'] = (btc['close'] - (sma - 2*std)) / (4*std).replace(0, np.nan) - 0.5

    # ── MACD ──────────────────────────────────────────────────────────────────
    for fast, slow, sig in [(12, 26, 9), (5, 13, 5)]:
        ema_f = btc['close'].ewm(span=fast, adjust=False).mean()
        ema_s = btc['close'].ewm(span=slow, adjust=False).mean()
        macd_ = ema_f - ema_s
        out[f'macd_{fast}_{slow}']      = macd_ / btc['close']
        out[f'macd_hist_{fast}_{slow}'] = (macd_ - macd_.ewm(span=sig, adjust=False).mean()) / btc['close']

    # ── RSI ───────────────────────────────────────────────────────────────────
    def _rsi(close, n):
        d  = close.diff()
        g  = d.clip(lower=0)
        l  = (-d).clip(lower=0)
        ag = g.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
        al = l.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
        return 100 - 100 / (1 + ag / (al + 1e-12))

    for n in [7, 14, 21]:
        out[f'rsi_{n}'] = _rsi(btc['close'], n)

    # ── Stochastic %K ─────────────────────────────────────────────────────────
    for n in [14, 21]:
        lo = btc['low'].rolling(n).min()
        hi = btc['high'].rolling(n).max()
        out[f'stoch_k_{n}'] = (btc['close'] - lo) / (hi - lo + 1e-12) * 100 - 50

    # ── Williams %R ───────────────────────────────────────────────────────────
    lo14 = btc['low'].rolling(14).min()
    hi14 = btc['high'].rolling(14).max()
    out['williams_r'] = (hi14 - btc['close']) / (hi14 - lo14 + 1e-12) * (-100)

    # ── Volume z-score and ratio ───────────────────────────────────────────────
    for h in [12, 24, 72, 168]:
        vm = btc['volume'].rolling(h).mean()
        vs = btc['volume'].rolling(h).std()
        out[f'vol_z_{h}h']     = (btc['volume'] - vm) / (vs + 1e-12)
        out[f'vol_ratio_{h}h'] = btc['volume'] / (vm + 1e-12)

    # ── OBV z-score ───────────────────────────────────────────────────────────
    obv = (np.sign(btc['close'].diff()) * btc['volume']).cumsum()
    out['obv_z_72'] = (obv - obv.rolling(72).mean()) / (obv.rolling(72).std() + 1e-12)

    # ── Candle anatomy ────────────────────────────────────────────────────────
    hl_range = btc['high'] - btc['low'] + 1e-12
    body      = (btc['close'] - btc['open']).abs() / hl_range
    out['candle_body']  = body
    out['upper_wick']   = (btc['high'] - btc[['open', 'close']].max(axis=1)) / hl_range
    out['lower_wick']   = (btc[['open', 'close']].min(axis=1) - btc['low']) / hl_range
    out['is_bullish']   = (btc['close'] > btc['open']).astype(float)

    # ── H/L position ──────────────────────────────────────────────────────────
    for h in [24, 48, 168]:
        lo = btc['low'].rolling(h).min()
        hi = btc['high'].rolling(h).max()
        out[f'hl_position_{h}h'] = (btc['close'] - lo) / (hi - lo + 1e-12)

    # ── Time features ─────────────────────────────────────────────────────────
    out['hour_sin']     = np.sin(2 * np.pi * btc.index.hour / 24)
    out['hour_cos']     = np.cos(2 * np.pi * btc.index.hour / 24)
    out['dow_sin']      = np.sin(2 * np.pi * btc.index.dayofweek / 7)
    out['dow_cos']      = np.cos(2 * np.pi * btc.index.dayofweek / 7)
    out['dom_sin']      = np.sin(2 * np.pi * btc.index.day / 31)
    out['dom_cos']      = np.cos(2 * np.pi * btc.index.day / 31)
    out['quarter_sin']  = np.sin(2 * np.pi * btc.index.quarter / 4)
    out['quarter_cos']  = np.cos(2 * np.pi * btc.index.quarter / 4)

    # ── MA cross signals ──────────────────────────────────────────────────────
    sma50  = btc['close'].rolling(50).mean()
    sma200 = btc['close'].rolling(200).mean()
    sma20  = btc['close'].rolling(20).mean()
    sma100 = btc['close'].rolling(100).mean()
    out['sma50_vs_sma200']  = (sma50 - sma200) / sma200.replace(0, np.nan)
    out['sma20_vs_sma50']   = (sma20 - sma50)  / sma50.replace(0, np.nan)
    out['sma50_vs_sma100']  = (sma50 - sma100) / sma100.replace(0, np.nan)
    out['sma100_vs_sma200'] = (sma100 - sma200) / sma200.replace(0, np.nan)
    gc = (sma50 > sma200).astype(int)
    out['golden_cross'] = gc

    # Proper candles_since_cross: count bars since last cross
    since    = np.zeros(len(btc))
    cnt      = 0
    gc_vals  = gc.values
    for i in range(1, len(gc_vals)):
        if gc_vals[i] != gc_vals[i-1]:
            cnt = 0
        else:
            cnt += 1
        since[i] = cnt
    out['candles_since_cross'] = since
    out['cross_recency']       = np.exp(-since / 168)

    # MA bull score: fraction of MA pairs where fast > slow
    ma_pairs = [(7,14),(7,50),(14,50),(20,50),(50,100),(50,200),(100,200)]
    scores = []
    for f, s in ma_pairs:
        fa = btc['close'].rolling(f).mean()
        sl = btc['close'].rolling(s).mean()
        scores.append((fa > sl).astype(float))
    out['ma_bull_score']   = pd.concat(scores, axis=1).mean(axis=1)
    out['ma_ribbon_width'] = (btc['close'].rolling(7).mean() - btc['close'].rolling(200).mean()) / btc['close']

    # ── Ichimoku ──────────────────────────────────────────────────────────────
    tenkan = (btc['high'].rolling(9).max()  + btc['low'].rolling(9).min())  / 2
    kijun  = (btc['high'].rolling(26).max() + btc['low'].rolling(26).min()) / 2
    span_a = ((tenkan + kijun) / 2).shift(26)
    span_b = ((btc['high'].rolling(52).max() + btc['low'].rolling(52).min()) / 2).shift(26)
    out['tk_ratio']          = (tenkan - kijun) / kijun.replace(0, np.nan)
    out['tk_cross_bull']     = ((tenkan > kijun) & (tenkan.shift(1) <= kijun.shift(1))).astype(float)
    out['tk_cross_bear']     = ((tenkan < kijun) & (tenkan.shift(1) >= kijun.shift(1))).astype(float)
    cloud_top = pd.concat([span_a, span_b], axis=1).max(axis=1)
    cloud_bot = pd.concat([span_a, span_b], axis=1).min(axis=1)
    out['close_vs_cloud_top']    = (btc['close'] - cloud_top) / cloud_top.replace(0, np.nan)
    out['close_vs_cloud_bottom'] = (btc['close'] - cloud_bot) / cloud_bot.replace(0, np.nan)
    out['inside_cloud']          = ((btc['close'] <= cloud_top) & (btc['close'] >= cloud_bot)).astype(float)
    out['cloud_thickness']       = (cloud_top - cloud_bot) / btc['close']
    out['cloud_bullish']         = (span_a > span_b).astype(float)
    flip = (out['cloud_bullish'].diff() != 0).astype(int)
    out['cloud_flip_recency']    = np.exp(-flip.cumsum() / 48)
    out['close_vs_kijun']        = (btc['close'] - kijun) / kijun.replace(0, np.nan)

    # ── Supertrend (3 ATR multipliers) ────────────────────────────────────────
    def _supertrend(close, high, low, atr_period, mult):
        atr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low  - close.shift(1)).abs(),
        ], axis=1).max(axis=1).ewm(span=atr_period, adjust=False).mean()
        ub = (high + low) / 2 + mult * atr
        lb = (high + low) / 2 - mult * atr
        trend = pd.Series(1, index=close.index)
        for i in range(1, len(close)):
            if   close.iloc[i] > ub.iloc[i-1]: trend.iloc[i] =  1
            elif close.iloc[i] < lb.iloc[i-1]: trend.iloc[i] = -1
            else:                               trend.iloc[i] =  trend.iloc[i-1]
        dist = (close - lb.where(trend == 1, ub)) / close
        return trend, dist

    for tag, mult in [('15', 1.5), ('20', 2.0), ('30', 3.0)]:
        t, d = _supertrend(btc['close'], btc['high'], btc['low'], 14, mult)
        out[f'supertrend_dir_{tag}']  = t.astype(float)
        out[f'supertrend_dist_{tag}'] = d
    out['supertrend_consensus'] = out[['supertrend_dir_15', 'supertrend_dir_20', 'supertrend_dir_30']].mean(axis=1)
    st_flip = (out['supertrend_dir_20'].diff() != 0).astype(int)
    out['supertrend_flip_recency'] = np.exp(-st_flip.cumsum() / 48)

    # ── Fibonacci retracements (48h and 168h windows) ──────────────────────────
    for h in [48, 168]:
        lo  = btc['low'].rolling(h).min()
        hi  = btc['high'].rolling(h).max()
        rng = (hi - lo).replace(0, np.nan)
        pos = (btc['close'] - lo) / rng
        out[f'fib_position_{h}h']      = pos
        lvl_618 = lo + 0.618 * rng
        out[f'fib_nearest_dist_{h}h']  = (btc['close'] - lvl_618) / rng
        out[f'fib_nearest_level_{h}h'] = 0.618
        out[f'fib_dist_618_{h}h']      = (btc['close'] - lvl_618) / rng
        out[f'fib_below_618_{h}h']     = (btc['close'] < lvl_618).astype(float)

    # ── Weekly momentum acceleration ───────────────────────────────────────────
    mom_24  = btc['close'].pct_change(24)
    mom_168 = btc['close'].pct_change(168)
    out['weekly_mom_accel'] = mom_24 - mom_168 / 7

    # ── Halving cycle ──────────────────────────────────────────────────────────
    HALVINGS = [
        pd.Timestamp('2012-11-28'), pd.Timestamp('2016-07-09'),
        pd.Timestamp('2020-05-11'), pd.Timestamp('2024-04-19'),
    ]
    PERIOD   = 365.25 * 24 * 3600
    t_since  = np.array([(ts - HALVINGS[-1]).total_seconds() for ts in btc.index])
    out['halving_cycle_sin'] = np.sin(2 * np.pi * t_since / PERIOD)
    out['halving_cycle_cos'] = np.cos(2 * np.pi * t_since / PERIOD)
    out['halving_cycle_pos'] = (t_since % PERIOD) / PERIOD

    # ── Divergences (price vs indicator) ──────────────────────────────────────
    def _divergence(price, indicator, w=24):
        p_high = price.rolling(w).max()
        p_low  = price.rolling(w).min()
        i_high = indicator.rolling(w).max()
        i_low  = indicator.rolling(w).min()
        bull = ((price <= p_low * 1.005) & (indicator > i_low * 1.05)).astype(float)
        bear = ((price >= p_high * 0.995) & (indicator < i_high * 0.95)).astype(float)
        return (bull - bear).rolling(3).mean()

    rsi14  = out['rsi_14']
    macd_s = out['macd_12_26']
    out['rsi_divergence']  = _divergence(btc['close'], rsi14)
    out['macd_divergence'] = _divergence(btc['close'], macd_s)
    out['obv_divergence']  = _divergence(btc['close'], obv)

    # ── Volume/price divergences ───────────────────────────────────────────────
    out['vol_price_div_12h'] = (
        out['vol_z_12h'] - log_ret.rolling(12).mean() / log_ret.rolling(12).std().replace(0, np.nan)
    )
    out['vol_price_div_24h'] = (
        out['vol_z_24h'] - log_ret.rolling(24).mean() / log_ret.rolling(24).std().replace(0, np.nan)
    )

    # ── Pattern recognition ────────────────────────────────────────────────────
    doji_size = (btc['close'] - btc['open']).abs() / (btc['high'] - btc['low'] + 1e-12)
    out['doji']          = (doji_size < 0.1).astype(float)
    out['hammer']        = ((out['lower_wick'] > 0.6) & (out['upper_wick'] < 0.1) & (btc['close'] > btc['open'])).astype(float)
    out['shooting_star'] = ((out['upper_wick'] > 0.6) & (out['lower_wick'] < 0.1) & (btc['close'] < btc['open'])).astype(float)
    out['bull_engulf']   = (
        (btc['close'] > btc['open']) & (btc['close'].shift(1) < btc['open'].shift(1)) &
        (btc['close'] > btc['open'].shift(1)) & (btc['open'] < btc['close'].shift(1))
    ).astype(float)
    out['bear_engulf']   = (
        (btc['close'] < btc['open']) & (btc['close'].shift(1) > btc['open'].shift(1)) &
        (btc['close'] < btc['open'].shift(1)) & (btc['open'] > btc['close'].shift(1))
    ).astype(float)
    out['marubozu']      = (body > 0.9).astype(float)

    bull_s  = np.zeros(len(btc))
    bear_s  = np.zeros(len(btc))
    is_up   = (btc['close'] > btc['open']).values
    for i in range(1, len(btc)):
        bull_s[i] = (bull_s[i-1] + 1) if is_up[i]     else 0
        bear_s[i] = (bear_s[i-1] + 1) if not is_up[i] else 0
    out['bull_streak'] = bull_s
    out['bear_streak'] = bear_s

    # ── VWAP ──────────────────────────────────────────────────────────────────
    pv24  = (btc['close'] * btc['volume']).rolling(24).sum()
    pv168 = (btc['close'] * btc['volume']).rolling(168).sum()
    out['close_vs_vwap_24h']  = (btc['close'] - pv24  / btc['volume'].rolling(24).sum().replace(0, np.nan))  / btc['close']
    out['close_vs_vwap_168h'] = (btc['close'] - pv168 / btc['volume'].rolling(168).sum().replace(0, np.nan)) / btc['close']

    # ── MFI, CMF, AD ──────────────────────────────────────────────────────────
    tp     = (btc['high'] + btc['low'] + btc['close']) / 3
    mf     = tp * btc['volume']
    pos_mf = mf.where(tp > tp.shift(1), 0).rolling(14).sum()
    neg_mf = mf.where(tp < tp.shift(1), 0).rolling(14).sum()
    out['mfi_14'] = 100 - 100 / (1 + pos_mf / (neg_mf + 1e-12))
    pos_mf21 = mf.where(tp > tp.shift(1), 0).rolling(21).sum()
    neg_mf21 = mf.where(tp < tp.shift(1), 0).rolling(21).sum()
    out['mfi_21'] = 100 - 100 / (1 + pos_mf21 / (neg_mf21 + 1e-12))
    clv = ((btc['close'] - btc['low']) - (btc['high'] - btc['close'])) / (btc['high'] - btc['low'] + 1e-12)
    out['cmf_20']    = (clv * btc['volume']).rolling(20).sum() / btc['volume'].rolling(20).sum().replace(0, np.nan)
    ad   = (clv * btc['volume']).cumsum()
    out['ad_z_48h']  = (ad - ad.rolling(48).mean())  / (ad.rolling(48).std()  + 1e-12)
    out['ad_z_168h'] = (ad - ad.rolling(168).mean()) / (ad.rolling(168).std() + 1e-12)

    # ── Volume spikes ──────────────────────────────────────────────────────────
    vol_mean_72 = btc['volume'].rolling(72).mean()
    out['vol_spike_2x'] = (btc['volume'] > 2 * vol_mean_72).astype(float)
    out['vol_spike_3x'] = (btc['volume'] > 3 * vol_mean_72).astype(float)

    # ── VW-RSI ────────────────────────────────────────────────────────────────
    vw_close = (btc['close'] * btc['volume']).rolling(14).sum() / btc['volume'].rolling(14).sum().replace(0, np.nan)
    out['vw_rsi_14'] = _rsi(vw_close, 14)

    # ── Pivot levels (previous-day H/L/C) ─────────────────────────────────────
    daily_hi = btc['high'].resample('D').max().shift(1, freq='D').reindex(btc.index, method='ffill')
    daily_lo = btc['low'].resample('D').min().shift(1, freq='D').reindex(btc.index, method='ffill')
    daily_cl = btc['close'].resample('D').last().shift(1, freq='D').reindex(btc.index, method='ffill')
    pivot = (daily_hi + daily_lo + daily_cl) / 3
    r1 = 2*pivot - daily_lo;    s1 = 2*pivot - daily_hi
    r2 = pivot + (daily_hi - daily_lo); s2 = pivot - (daily_hi - daily_lo)
    out['close_vs_pivot'] = (btc['close'] - pivot) / btc['close']
    out['close_vs_r1']    = (btc['close'] - r1)    / btc['close']
    out['close_vs_s1']    = (btc['close'] - s1)    / btc['close']
    out['close_vs_r2']    = (btc['close'] - r2)    / btc['close']
    out['close_vs_s2']    = (btc['close'] - s2)    / btc['close']

    # ── Round number distances ─────────────────────────────────────────────────
    out['dist_round_1000']  = ((btc['close'] % 1000)  / 1000  - 0.5).abs()
    out['dist_round_10000'] = ((btc['close'] % 10000) / 10000 - 0.5).abs()

    # ── Breakouts ─────────────────────────────────────────────────────────────
    for h in [24, 48, 168]:
        hi = btc['high'].rolling(h).max().shift(1)
        lo = btc['low'].rolling(h).min().shift(1)
        out[f'breakout_up_{h}h']   = (btc['close'] > hi).astype(float)
        out[f'breakout_down_{h}h'] = (btc['close'] < lo).astype(float)

    # ── Garman-Klass and Parkinson volatility ──────────────────────────────────
    for h in [24, 72, 168]:
        gk = (
            0.5 * np.log(btc['high'] / btc['low'])**2
            - (2*np.log(2) - 1) * np.log(btc['close'] / btc['open'])**2
        ).rolling(h).mean()
        out[f'gk_vol_{h}h'] = np.sqrt(gk.clip(lower=0))
    for h in [24, 72]:
        pk = (0.5 * np.log(btc['high'] / btc['low'])**2).rolling(h).mean()
        out[f'park_vol_{h}h'] = np.sqrt(pk.clip(lower=0))

    # ── Vol of vol, ATR rank, BB squeeze ──────────────────────────────────────
    out['vol_of_vol_72h']  = out['vol_24h'].rolling(72).std()
    out['atr_14_pct_rank'] = out['atr_14_pct'].rolling(168).rank(pct=True)
    out['atr_24_pct_rank'] = out['atr_24_pct'].rolling(168).rank(pct=True)
    out['bb_squeeze_20']   = (out['bb_width_20'] < out['bb_width_20'].rolling(20).quantile(0.25)).astype(float)
    out['bb_squeeze_50']   = (out['bb_width_50'] < out['bb_width_50'].rolling(50).quantile(0.25)).astype(float)
    out['range_vs_atr']    = (btc['high'] - btc['low']) / out['atr_14'].replace(0, np.nan)

    # ── Hurst exponent (R/S, rolling 168h) ────────────────────────────────────
    def _hurst(series, w=168):
        result = np.full(len(series), np.nan)
        arr = series.values
        for i in range(w-1, len(arr)):
            seg = arr[i-w+1:i+1]
            if np.any(np.isnan(seg)):
                continue
            m   = seg.mean()
            dev = np.cumsum(seg - m)
            R   = dev.max() - dev.min()
            S   = seg.std()
            if S > 0:
                result[i] = np.log(R / S) / np.log(w)
        return pd.Series(result, index=series.index)

    out['hurst_168h'] = _hurst(log_ret, 168)

    # ── Skewness, kurtosis ────────────────────────────────────────────────────
    for h in [24, 72, 168]:
        out[f'skew_{h}h'] = log_ret.rolling(h).skew()
        out[f'kurt_{h}h'] = log_ret.rolling(h).kurt()

    # ── Autocorrelation ───────────────────────────────────────────────────────
    for h in [1, 6, 12, 24]:
        out[f'autocorr_ret_{h}h'] = log_ret.rolling(48).corr(log_ret.shift(h))

    # ── Variance ratio (test for random walk) ─────────────────────────────────
    for h in [6, 24]:
        var_h = log_ret.rolling(h).var()
        var_1 = log_ret.rolling(1).var()
        out[f'var_ratio_{h}h'] = (var_h / (h * var_1 + 1e-12)).clip(0, 5)

    # ── Trend score (MA alignment 0–1) ────────────────────────────────────────
    ma_cols = ['close_vs_sma_7', 'close_vs_sma_20', 'close_vs_sma_50', 'close_vs_sma_200']
    out['trend_score'] = out[ma_cols].apply(np.sign).mean(axis=1)

    # ── RSI + volume confirmation ─────────────────────────────────────────────
    out['rsi_vol_confirm'] = out['rsi_14'] / 100 * out['vol_ratio_24h'].clip(0, 3) / 3

    # ── Momentum coherence ────────────────────────────────────────────────────
    ret_cols = [f'ret_{h}h' for h in [1, 2, 3, 6, 12, 24]]
    out['mom_coherence'] = out[ret_cols].apply(np.sign).mean(axis=1)

    # ── Rolling Sharpe ────────────────────────────────────────────────────────
    for h in [24, 72, 168]:
        r = log_ret.rolling(h)
        out[f'sharpe_ratio_{h}h'] = (r.mean() / (r.std() + 1e-12)) * np.sqrt(24 * 365)

    # ── Regime composite ──────────────────────────────────────────────────────
    out['regime_composite'] = (
        out['trend_score'] * 0.4
        + out['mom_coherence'] * 0.3
        + (out['hurst_168h'] - 0.5) * 0.3
    ).clip(-1, 1)

    # ── Label (next-bar direction) ─────────────────────────────────────────────
    out['label'] = (btc['close'].shift(-1) > btc['close']).astype(int)

    out.to_parquet(V1_OUTPUT)
    print(f'  V1 features saved: {out.shape}  →  {V1_OUTPUT.name}')


  SKIP V1 technical features  (105.5 MB)


---
## Phase E — V3 External Features (with Lag Fix)

Builds 21 features from external data (Fear & Greed, market caps, cross-asset).

### Critical Lag Fix

Daily-frequency external data is an **end-of-day** snapshot. Without a 1-day forward shift,
bar `D` would see the *current* day's market cap change, leaking forward-looking information.

The `mkt_total_mcap_chg_24h` feature had `corr = +0.18` with next-bar returns before this fix.
Every daily series is now shifted by 1 day BEFORE `.reindex(btc.index, method='ffill')`.

Cross-asset and intraday-computed features (e.g. `cross_eth_btc_ratio`) do NOT need the shift
since they are computed at hourly granularity.

Output: `data/features/BTCUSDT_1h_v3_features.parquet`


In [42]:

# LAG FIX: daily external data is an end-of-day value.
# It must be shifted forward by 1 day before forward-filling to hourly bars,
# so that bars on day D only see the value from day D-1.
# Without this shift, mkt_total_mcap_chg_24h correlates +0.18 with next-bar return.

V3_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet'

if _skip(V3_OUTPUT, 'V3 external features (lag-fixed)'):
    pass
else:
    print('Phase E — Building V3 external features with lag fix ...')

    # ── Load base data ────────────────────────────────────────────────────────
    btc = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet')
    btc.index = btc.index.tz_localize(None) if btc.index.tz else btc.index

    alt_closes = {}
    for sym in ALTS:
        p = RAW_DIR / f'{sym}_1h.parquet'
        if p.exists():
            df = pd.read_parquet(p)
            df.index = df.index.tz_localize(None) if df.index.tz else df.index
            alt_closes[sym] = df['close'].reindex(btc.index, method='ffill')

    fng_df = pd.read_parquet(EXT_DIR / 'fear_greed_index.parquet')
    fng_df.index = fng_df.index.tz_localize(None) if fng_df.index.tz else fng_df.index

    approx_mcap = pd.read_parquet(EXT_DIR / 'approx_market_caps.parquet')
    approx_mcap['date'] = pd.to_datetime(approx_mcap['date']).dt.tz_localize(None)

    cg_mcap = pd.read_parquet(EXT_DIR / 'coingecko_market_caps.parquet')
    cg_mcap['date'] = pd.to_datetime(cg_mcap['date']).dt.tz_localize(None)

    print(f'  BTC: {len(btc):,} bars  |  Alts: {list(alt_closes.keys())}')

    v3 = pd.DataFrame(index=btc.index)

    # ── Group 1: Cross-Asset (hourly, NO lag needed) ───────────────────────────
    eth = alt_closes.get('ETHUSDT')
    if eth is not None:
        eth_btc = eth / btc['close']
        v3['cross_eth_btc_ratio']   = eth_btc
        v3['cross_eth_btc_mom_24h'] = eth_btc.pct_change(24)
        v3['cross_eth_btc_mom_72h'] = eth_btc.pct_change(72)

    alt_rets = pd.DataFrame({sym: c.pct_change(24) for sym, c in alt_closes.items()})
    if not alt_rets.empty:
        btc_ret_24h = btc['close'].pct_change(24)
        avg_alt_ret = alt_rets.mean(axis=1)
        v3['cross_altcoin_breadth_24h']   = (alt_rets > 0).mean(axis=1)
        v3['cross_btc_relative_strength'] = btc_ret_24h - avg_alt_ret
        v3['cross_alt_correlation_24h']   = btc_ret_24h.rolling(24).corr(avg_alt_ret)
    print(f'  Group 1 (cross-asset): {sum(c.startswith("cross_") for c in v3.columns)} features')

    # ── Group 2: Market Structure (daily → LAG FIX applied) ───────────────────
    mcap_pivot = approx_mcap.pivot_table(
        index='date', columns='symbol', values='approx_market_cap', aggfunc='last'
    )
    our_coins   = [s for s in SYMBOLS_1H if s in mcap_pivot.columns]
    total_mcap  = mcap_pivot[our_coins].sum(axis=1)
    btc_dom     = mcap_pivot.get('BTCUSDT', pd.Series(dtype=float)) / total_mcap
    eth_dom     = mcap_pivot.get('ETHUSDT', pd.Series(dtype=float)) / total_mcap

    cg_stable = cg_mcap[cg_mcap['symbol'].isin(['USDTUSDT', 'USDCUSDT'])]
    if not cg_stable.empty:
        stable_daily = cg_stable.pivot_table(
            index=cg_stable['date'].dt.normalize(), columns='symbol',
            values='market_cap', aggfunc='last',
        ).sum(axis=1)
        stable_reindx  = stable_daily.reindex(mcap_pivot.index, method='ffill')
        stablecoin_pct = stable_reindx / (total_mcap + stable_reindx).replace(0, np.nan)
    else:
        stablecoin_pct = pd.Series(np.nan, index=mcap_pivot.index)

    mkt_daily = pd.DataFrame({
        'mkt_btc_dominance':        btc_dom,
        'mkt_btc_dominance_chg_7d': btc_dom.diff(7),
        'mkt_eth_dominance':        eth_dom,
        'mkt_total_mcap_chg_24h':   total_mcap.pct_change(1),
        'mkt_stablecoin_pct':       stablecoin_pct,
    })
    # LAG FIX: shift daily series 1 day before reindexing to hourly
    for col in mkt_daily.columns:
        v3[col] = mkt_daily[col].shift(1).reindex(btc.index, method='ffill')  # 1-day lag
    print(f'  Group 2 (market structure): {sum(c.startswith("mkt_") for c in v3.columns)} features')

    # ── Group 3: Sentiment (daily → LAG FIX applied) ──────────────────────────
    fng_scaled = fng_df[['value']].rename(columns={'value': 'sent_fear_greed'})
    fng_scaled['sent_fear_greed']        = fng_scaled['sent_fear_greed'] / 100.0
    fng_scaled['sent_fear_greed_ma7']    = fng_scaled['sent_fear_greed'].rolling(7).mean()
    fng_scaled['sent_fear_greed_chg_7d'] = fng_scaled['sent_fear_greed'].diff(7)
    # LAG FIX: shift daily sentiment 1 day before reindexing
    for col in fng_scaled.columns:
        v3[col] = fng_scaled[col].shift(1).reindex(btc.index, method='ffill')  # 1-day lag
    print(f'  Group 3 (sentiment): {sum(c.startswith("sent_") for c in v3.columns)} features')

    # ── Group 4: Microstructure (hourly, NO lag needed) ───────────────────────
    close      = btc['close']
    volume     = btc['volume']
    dollar_vol = close * volume
    ret        = close.pct_change()
    MICRO_WIN  = 24

    v3['micro_amihud_illiq'] = (
        ret.abs() / dollar_vol.replace(0, np.nan)
    ).rolling(MICRO_WIN).mean()

    # Kyle's lambda: rolling OLS of |Δprice| on volume
    print('  Computing Kyle lambda (~30 s) ...', flush=True)
    delta_p = close.diff().abs()
    kl      = np.full(len(btc), np.nan)
    dp_arr  = delta_p.values
    v_arr   = volume.values
    for i in range(MICRO_WIN, len(btc)):
        dp   = dp_arr[i - MICRO_WIN:i]
        v    = v_arr[i - MICRO_WIN:i]
        mask = np.isfinite(dp) & np.isfinite(v) & (v > 0)
        if mask.sum() > 5 and v[mask].std() > 0:
            kl[i] = np.polyfit(v[mask], dp[mask], 1)[0]
    v3['micro_kyle_lambda'] = pd.Series(kl, index=btc.index)

    cov_serial = ret.rolling(MICRO_WIN).cov(ret.shift(1))
    v3['micro_roll_spread'] = pd.Series(
        np.where(cov_serial < 0, np.sqrt(-2 * cov_serial), 0.0),
        index=btc.index,
    )
    vol_mean = volume.rolling(MICRO_WIN).mean()
    vol_std  = volume.rolling(MICRO_WIN).std()
    v3['micro_volume_clock'] = vol_std / vol_mean.replace(0, np.nan)
    print(f'  Group 4 (microstructure): {sum(c.startswith("micro_") for c in v3.columns)} features')

    # ── Group 5: Enhanced OHLCV (hourly, NO lag needed) ───────────────────────
    vol_24  = ret.rolling(24).std()
    vol_168 = ret.rolling(168).std()
    v3['vol_term_structure'] = vol_24 / vol_168.replace(0, np.nan)
    v3['mom_normalized_24h'] = close.pct_change(24)  / vol_24.replace(0, np.nan)
    v3['mom_normalized_72h'] = close.pct_change(72)  / vol_168.replace(0, np.nan)
    print(f'  Group 5 (enhanced OHLCV): {sum(c.startswith("vol_term") or c.startswith("mom_norm") for c in v3.columns)} features')

    v3.to_parquet(V3_OUTPUT)
    print(f'\n  Total V3 features: {v3.shape[1]}')
    print(f'  Date range: {v3.index.min().date()} → {v3.index.max().date()}')
    print(f'  Saved: {V3_OUTPUT}')


Phase E — Building V3 external features with lag fix ...
  BTC: 76,938 bars  |  Alts: ['ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT', 'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT', 'DOTUSDT', 'LINKUSDT']
  Group 1 (cross-asset): 6 features
  Group 2 (market structure): 5 features
  Group 3 (sentiment): 3 features
  Computing Kyle lambda (~30 s) ...
  Group 4 (microstructure): 4 features
  Group 5 (enhanced OHLCV): 3 features

  Total V3 features: 21
  Date range: 2017-08-17 → 2026-05-27
  Saved: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/features/BTCUSDT_1h_v3_features.parquet


---
## Phase F — V4 Features

Builds the V4 feature set from V1 + V3 base data plus microstructure feeds.

Feature groups:
- Fractional differencing (`fracdiff_close_d{d}`)
- Regime: Hurst (24h, 72h), ADF stat/pval at multiple windows, BB squeeze, sideways flag
- Trade Flow Imbalance (TFI): `tfi_pct`, `tfi_z_*`, `tfi_ema_*`
- Average trade size (institutional signature)
- True VWAP from quote volume
- Taker price premium (aggressor urgency)
- Futures mark price basis and z-score
- Index price deviation and lead/lag

Requires `hmats.data.v4_features` and `hmats.data.fetch_taker_volume`.

Output: `data/features/BTCUSDT_1h_v4_features.parquet`


In [43]:

import sys
sys.path.insert(0, str(REPO / 'src'))

V4_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet'

if _skip(V4_OUTPUT, 'V4 features'):
    pass
else:
    print('Phase F — Building V4 features ...')
    from hmats.data.v4_features import build_v4_features
    from hmats.data.fetch_taker_volume import load_taker_volume

    btc   = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet')
    btc.index = btc.index.tz_localize(None) if btc.index.tz else btc.index

    v3_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet')
    v3_df.index = v3_df.index.tz_localize(None) if v3_df.index.tz else v3_df.index

    taker = None
    taker_path = RAW_DIR / 'BTCUSDT_1h_taker.parquet'
    if taker_path.exists():
        taker = load_taker_volume(store_dir=str(RAW_DIR))
        print(f'  Taker volume enabled: {len(taker):,} rows')
    else:
        print('  Taker volume not found — TFI features will be skipped')

    v4 = build_v4_features(btc, taker_df=taker, v3_df=v3_df, verbose=True)
    v4.to_parquet(V4_OUTPUT)
    print(f'  V4 features: {v4.shape}  →  {V4_OUTPUT.name}')


  SKIP V4 features  (17.0 MB)


---
## Phase G — Structural Features

Builds 39 physics-inspired structural features from BTCUSDT 1h OHLCV via an external script.

Feature groups:
- Swing extrema (confirmed minor/major highs & lows, proximity, age)
- Liquidity proxies (anchored VWAP, rolling POC, volume z-score, exhaustion spikes)
- Volatility (ATR, Bollinger Bands, Keltner squeeze, Garman-Klass)
- Multi-timeframe context (4h and daily EMA spread, RSI, composite alignment)

Output: `data/features/BTCUSDT_1h_structural.parquet`


In [44]:

STRUCT_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_structural.parquet'

if _skip(STRUCT_OUTPUT, 'Structural features'):
    pass
else:
    import subprocess, sys
    print('Phase G — Building structural features via external script ...')
    result = subprocess.run(
        [sys.executable, str(REPO / 'src/hmats/notebooks/01_structural_features_1h.py')],
        capture_output=True,
        text=True,
        cwd=str(REPO),
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    else:
        print('Phase G — done.')


  SKIP Structural features  (12.6 MB)


---
## Phase H — Microstructure Features

Builds four stationary, strictly-causal microstructure features.

| Feature | Meaning |
|---------|---------|
| `roll_measure_50` | Effective bid-ask spread (Roll 1984) on log returns |
| `amihud_50` | Illiquidity / price impact per unit volume (Amihud 2002) |
| `vol_imbalance_50` | Taker buy/sell flow imbalance ∈ [0, 1] |
| `sampen_48` | Sample Entropy of 48h log-return path |

Every output is `.shift(1)`-ed so bar `t` only sees information through `t-1`.

Requires `hmats.features.microstructure` and `data/raw/BTCUSDT_1h_taker.parquet`.

Output: `data/features/BTCUSDT_1h_microstructure.parquet`


In [45]:

MICRO_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet'

if _skip(MICRO_OUTPUT, 'Microstructure features'):
    pass
else:
    import sys
    sys.path.insert(0, str(REPO / 'src'))
    from hmats.features.microstructure import add_microstructure_features

    print('Phase H — Building causal microstructure features ...')
    taker_path = RAW_DIR / 'BTCUSDT_1h_taker.parquet'
    if not taker_path.exists():
        print('  SKIP — taker volume parquet not found; run Phase F (taker download) first.')
    else:
        _tk = pd.read_parquet(taker_path)
        _tk.index = _tk.index.tz_localize(None) if _tk.index.tz else _tk.index
        _src = _tk[['close', 'volume', 'taker_buy_base_volume']].copy()

        micro = add_microstructure_features(_src, w=50, entropy_w=48)
        micro = micro[['roll_measure_50', 'amihud_50', 'vol_imbalance_50', 'sampen_48']]
        micro.to_parquet(MICRO_OUTPUT)
        print(f'  Microstructure: {micro.shape}  →  {MICRO_OUTPUT.name}')
        print(f'  Warm-up NaNs: {micro.isna().sum().to_dict()}')


  SKIP Microstructure features  (2.8 MB)


---
## Phase I — Merge to Unified Parquet

Merges all feature sources into a single parquet.

| Source | Role |
|--------|------|
| V1 features | Master index, technical features, `close`, `label` |
| Raw OHLCV | Supplies `open`, `high`, `low`, `volume` |
| V3 features | External features with lag fix |
| V4 features | Fractional diff, TFI, futures basis |
| Structural | Swing extrema, VWAP, MTF |
| Microstructure | Roll, Amihud, Sample Entropy |

All sources are `.reindex(master_index)` to align with the V1 index.

Output: `data/features/BTCUSDT_1h_unified.parquet`


In [46]:

UNIFIED_OUTPUT = FEAT_DIR / 'BTCUSDT_1h_unified.parquet'

if _skip(UNIFIED_OUTPUT, 'Unified parquet'):
    unified = pd.read_parquet(UNIFIED_OUTPUT)
    print(f'  Loaded: {unified.shape}')
else:
    print('Phase I — Merging all feature sources ...')

    # V1 is the master index
    v1 = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
    v1.index = v1.index.tz_localize(None) if v1.index.tz else v1.index
    master_index = v1.index
    unified = v1.copy()
    print(f'  V1 (master): {unified.shape}  {master_index.min().date()} → {master_index.max().date()}')

    # Raw OHLCV — supply open/high/low/volume
    raw = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet')
    raw.index = raw.index.tz_convert(None) if raw.index.tz else raw.index
    for col in ['open', 'high', 'low', 'volume']:
        unified[col] = raw[col].reindex(master_index)
    print(f'  Raw OHLCV: added open/high/low/volume')

    # V3 external features
    v3_path = FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet'
    if v3_path.exists():
        v3 = pd.read_parquet(v3_path)
        v3.index = v3.index.tz_localize(None) if v3.index.tz else v3.index
        for col in v3.columns:
            unified[col] = v3[col].reindex(master_index)
        print(f'  V3 external: {len(v3.columns)} cols merged')
    else:
        print('  V3 external: NOT FOUND — skipped')

    # V4 features
    v4_path = FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet'
    if v4_path.exists():
        v4 = pd.read_parquet(v4_path)
        v4.index = v4.index.tz_localize(None) if v4.index.tz else v4.index
        for col in v4.columns:
            unified[col] = v4[col].reindex(master_index)
        print(f'  V4 features: {len(v4.columns)} cols merged')
    else:
        print('  V4 features: NOT FOUND — skipped')

    # Structural features
    struct_path = FEAT_DIR / 'BTCUSDT_1h_structural.parquet'
    if struct_path.exists():
        struct = pd.read_parquet(struct_path)
        struct.index = struct.index.tz_convert(None) if struct.index.tz else struct.index
        for col in struct.columns:
            unified[col] = struct[col].reindex(master_index)
        print(f'  Structural: {len(struct.columns)} cols merged')
    else:
        print('  Structural: NOT FOUND — skipped')

    # Microstructure features
    micro_path = FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet'
    if micro_path.exists():
        micro = pd.read_parquet(micro_path)
        micro.index = micro.index.tz_localize(None) if micro.index.tz else micro.index
        for col in micro.columns:
            unified[col] = micro[col].reindex(master_index)
        print(f'  Microstructure: {len(micro.columns)} cols merged')
    else:
        print('  Microstructure: NOT FOUND — skipped')

    unified.to_parquet(UNIFIED_OUTPUT)
    mb = UNIFIED_OUTPUT.stat().st_size / 1e6
    print(f'\n  Merged shape: {unified.shape}')
    print(f'  Range: {unified.index.min().date()} → {unified.index.max().date()}')
    print(f'  Saved: {UNIFIED_OUTPUT.name}  ({mb:.1f} MB)')
    print(f'  Label distribution: {unified["label"].value_counts().to_dict()}')

print(f'\nUnified columns: {len(unified.columns)}')


  SKIP Unified parquet  (144.9 MB)
  Loaded: (74366, 292)

Unified columns: 292


---
## Phase J — Leakage Validation

Computes the correlation of every feature column with the next-bar log return.
Flags any feature with `|corr| > 0.05` as a potential data leak.

A legitimate feature should have `|corr| < 0.05` with the label — anything higher
likely reflects look-ahead bias (e.g. using current-bar close to compute a
`next_bar_return` target while also including it in a feature).


In [47]:

print('Phase J — Leakage validation ...')

# Load unified if not already in memory
if 'unified' not in dir() or unified is None:
    unified = pd.read_parquet(UNIFIED_OUTPUT)

# Next-bar log return = what the model is trying to predict
next_ret = np.log(unified['close'] / unified['close'].shift(1)).shift(-1)

# Skip non-numeric and the target itself
exclude  = {'label', 'close'}
feature_cols = [c for c in unified.columns
                if c not in exclude and pd.api.types.is_numeric_dtype(unified[c])]

leaks = []
for col in feature_cols:
    c = unified[col].corr(next_ret)
    if abs(c) > 0.05:
        leaks.append((col, c))
        print(f'  WARN {col}: corr={c:+.4f}')

if not leaks:
    print('No leakage detected (all |corr| < 0.05)')
else:
    print(f'\n{len(leaks)} potential leaks found — review the list above.')

print(f'\nTotal features checked: {len(feature_cols)}')


Phase J — Leakage validation ...
No leakage detected (all |corr| < 0.05)

Total features checked: 290


---
## Phase K — Update Feature Registry

Updates `data/features/feature_registry_unified_v1.json` with the current shape,
column list, and timestamp. Marks the V3 section with `"lag_fix_applied": true`.


In [48]:

import json
from datetime import datetime, timezone

REGISTRY_PATH = FEAT_DIR / 'feature_registry_unified_v1.json'

print('Phase K — Updating feature registry ...')

if 'unified' not in dir() or unified is None:
    unified = pd.read_parquet(UNIFIED_OUTPUT)

# Build source column sets for group accounting
v1_cols    = list(pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet').columns) if (FEAT_DIR / 'BTCUSDT_1h_features.parquet').exists() else []
v3_cols    = list(pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet').columns) if (FEAT_DIR / 'BTCUSDT_1h_v3_features.parquet').exists() else []
v4_cols    = list(pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet').columns) if (FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet').exists() else []
struct_cols= list(pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_structural.parquet').columns)  if (FEAT_DIR / 'BTCUSDT_1h_structural.parquet').exists() else []
micro_cols = list(pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet').columns) if (FEAT_DIR / 'BTCUSDT_1h_microstructure.parquet').exists() else []

registry = {
    "name":        "BTCUSDT_1h_unified_v1",
    "output_file": "data/features/BTCUSDT_1h_unified.parquet",
    "built_at":    datetime.now(timezone.utc).isoformat(),
    "shape": {
        "rows":    int(len(unified)),
        "columns": int(len(unified.columns)),
    },
    "date_range": {
        "start": str(unified.index.min().date()),
        "end":   str(unified.index.max().date()),
    },
    "sources": {
        "v1_technical": {
            "file":      "data/features/BTCUSDT_1h_features.parquet",
            "n_columns": len(v1_cols),
            "columns":   v1_cols,
        },
        "v3_external": {
            "file":             "data/features/BTCUSDT_1h_v3_features.parquet",
            "n_columns":        len(v3_cols),
            "columns":          v3_cols,
            "lag_fix_applied":  True,
            "lag_fix_note":     "Daily data shifted 1 day forward before ffill to hourly. "
                                "Fixes mkt_total_mcap_chg_24h leakage (was corr=+0.18).",
        },
        "v4_fractional_regime_tfi": {
            "file":      "data/features/BTCUSDT_1h_v4_features.parquet",
            "n_columns": len(v4_cols),
            "columns":   v4_cols,
        },
        "structural": {
            "file":      "data/features/BTCUSDT_1h_structural.parquet",
            "n_columns": len(struct_cols),
            "columns":   struct_cols,
        },
        "microstructure": {
            "file":      "data/features/BTCUSDT_1h_microstructure.parquet",
            "n_columns": len(micro_cols),
            "columns":   micro_cols,
        },
        "raw_ohlcv": {
            "file":    "data/raw/BTCUSDT_1h.parquet",
            "columns": ["open", "high", "low", "volume"],
        },
    },
    "all_columns": list(unified.columns),
}

with open(REGISTRY_PATH, 'w') as f:
    json.dump(registry, f, indent=2)

print(f'  Registry written: {REGISTRY_PATH.name}')
print(f'  Rows: {registry["shape"]["rows"]:,}')
print(f'  Columns: {registry["shape"]["columns"]}')
print(f'  V3 lag_fix_applied: {registry["sources"]["v3_external"]["lag_fix_applied"]}')
print(f'\nPipeline complete. Load unified parquet in model notebooks:')
print(f'  df = pd.read_parquet(FEAT_DIR / "BTCUSDT_1h_unified.parquet")')


Phase K — Updating feature registry ...
  Registry written: feature_registry_unified_v1.json
  Rows: 74,366
  Columns: 292
  V3 lag_fix_applied: True

Pipeline complete. Load unified parquet in model notebooks:
  df = pd.read_parquet(FEAT_DIR / "BTCUSDT_1h_unified.parquet")
